### old stuff

In [3]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/CEAS_08.csv")

print(df.shape)
df.head()

(39154, 7)


,sender,receiver,date,subject,body,label,urls
0,Young Esposito <Young@iworld.de>,user4@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 16:31:02 -0700",Never agree to be a loser,"Buck up, your troubles caused by small dimensi...",1,1
1,Mok <ipline's1983@icable.ph>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 18:31:03 -0500",Befriend Jenna Jameson,\nUpgrade your sex and pleasures with these te...,1,1
2,Daily Top 10 <Karmandeep-opengevl@universalnet...,user2.9@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 20:28:00 -1200",CNN.com Daily Top 10,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,1,1
3,Michael Parker <ivqrnai@pobox.com>,SpamAssassin Dev <xrh@spamassassin.apache.org>,"Tue, 05 Aug 2008 17:31:20 -0600",Re: svn commit: r619753 - in /spamassassin/tru...,Would anyone object to removing .so from this ...,0,1
4,Gretchen Suggs <externalsep1@loanofficertool.com>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 19:31:21 -0400",SpecialPricesPharmMoreinfo,\nWelcomeFastShippingCustomerSupport\nhttp://7...,1,1


### Accessing Files from Google Drive

Since the file `CEAS_08.csv` was not found, and you mentioned you'd download it from Drive, let's mount your Google Drive so Colab can access it. Then, you can provide the correct path to your CSV file.

In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


After running the cell above and following the authentication steps, your Google Drive will be mounted at `/content/drive`.

You can then locate your `CEAS_08.csv` file within your Google Drive. For example, if it's directly in your 'MyDrive' folder, the path would be `/content/drive/MyDrive/CEAS_08.csv`. If it's in a subfolder like 'Colab Notebooks', the path might be `/content/drive/MyDrive/Colab Notebooks/CEAS_08.csv`.

Please update the `pd.read_csv` line in the original cell (`hcp6CVZ00bRs`) with the correct path to your file in Google Drive, then re-run that cell.

In [4]:
print(df.columns.tolist())
print(df.dtypes)


['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls']
sender      object
receiver    object
date        object
subject     object
body        object
label        int64
urls         int64
dtype: object


In [5]:
print(df['label'].value_counts())


label
1    21842
0    17312
Name: count, dtype: int64


In [47]:
df["combined_text"] = df["subject"].fillna("") + " " + df["body"].fillna("")
print(df["combined_text"].head())


0    Never agree to be a loser Buck up, your troubl...
1    Befriend Jenna Jameson \nUpgrade your sex and ...
2    CNN.com Daily Top 10 >+=+=+=+=+=+=+=+=+=+=+=+=...
3    Re: svn commit: r619753 - in /spamassassin/tru...
4    SpecialPricesPharmMoreinfo \nWelcomeFastShippi...
Name: combined_text, dtype: object


In [58]:
import pandas as pd

train_df = pd.read_csv("/content/drive/MyDrive/spam_train_cleaned.csv")
test_df = pd.read_csv("/content/drive/MyDrive/spam_test_cleaned.csv")

print("Train:", train_df.shape)
print("Test:", test_df.shape)



Train: (31311, 16)
Test: (7828, 16)


In [63]:
same = set(train_df["combined_text"]) & set(test_df["combined_text"])

test_df = test_df[
    ~test_df["combined_text"].isin(same)
].reset_index(drop=True)

print("Train:", train_df.shape)
print("Test:", test_df.shape)

print("Same texts in Train & Test:",
      len(set(train_df["combined_text"]) & set(test_df["combined_text"])))



Train: (27411, 16)
Test: (6647, 16)
Same texts in Train & Test: 0


In [64]:
same = set(train_df["combined_text"]) & set(test_df["combined_text"])

print("Same texts in Train & Test:", len(same))


Same texts in Train & Test: 0


In [44]:
train_texts = set(train_df["combined_text"])

test_df = test_df[
    ~test_df["combined_text"].isin(train_texts)
].reset_index(drop=True)

print("Same texts in Train & Test:",
      len(set(train_df["combined_text"]) & set(test_df["combined_text"])))


Same texts in Train & Test: 0


In [45]:
# check class balance
print("Train class distribution:")
print(train_df["label"].value_counts())

print("\nTest class distribution:")
print(test_df["label"].value_counts())

Train class distribution:
label
1    17461
0    13850
Name: count, dtype: int64

Test class distribution:
label
0    3434
1    3241
Name: count, dtype: int64


In [46]:
print(train_df.columns.tolist())



['label', 'urls', 'hour', 'combined_text', 'capital_letter_count', 'capital_ratio', 'exclamation_count', 'question_count', 'special_char_count', 'day_of_week_Friday', 'day_of_week_Monday', 'day_of_week_Saturday', 'day_of_week_Sunday', 'day_of_week_Thursday', 'day_of_week_Tuesday', 'day_of_week_Wednesday']


In [65]:
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=300)

X_train = tfidf.fit_transform(train_df["combined_text"])
X_test = tfidf.transform(test_df["combined_text"])

y_train = train_df["label"]
y_test = test_df["label"]

print("Train:", X_train.shape)
print("Test:", X_test.shape)


Train: (27411, 300)
Test: (6647, 300)


### end of old stuff
### Start of new code fixing merge of features + adding cross valdetion

In [1]:
import pandas as pd
import numpy as np

train_df = pd.read_csv("spam_train_cleaned.csv")
test_df = pd.read_csv("spam_test_cleaned.csv")

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

print("\nTrain columns:")
print(train_df.columns.tolist())

display(train_df.head())

Train shape: (27246, 16)
Test shape : (6812, 16)

Train columns:
['label', 'urls', 'hour', 'combined_text', 'capital_letter_count', 'capital_ratio', 'exclamation_count', 'question_count', 'special_char_count', 'day_of_week_Friday', 'day_of_week_Monday', 'day_of_week_Saturday', 'day_of_week_Sunday', 'day_of_week_Thursday', 'day_of_week_Tuesday', 'day_of_week_Wednesday']


,label,urls,hour,combined_text,capital_letter_count,capital_ratio,exclamation_count,question_count,special_char_count,day_of_week_Friday,day_of_week_Monday,day_of_week_Saturday,day_of_week_Sunday,day_of_week_Thursday,day_of_week_Tuesday,day_of_week_Wednesday
0,0,1,9,re sm users sendmail verbose output date tue 1...,133,0.033739,1,0,244,0,0,0,0,1,0,0
1,0,1,21,bug 5695 flock interrupted by system call addi...,41,0.034746,0,1,7,0,0,0,0,0,0,1
2,1,1,20,to men who want to buy rolex watches at a frac...,26,0.120930,0,0,0,0,0,0,0,1,0,0
3,0,1,23,last day free overnight shipping fragrances fo...,206,0.111051,0,1,30,0,0,0,0,0,1,0
4,1,0,19,durham deity micron micron berth excellent cap...,0,0.000000,0,3,0,0,0,0,0,0,1,0


In [2]:
#Check duplicates
print("Train duplicates:", train_df["combined_text"].duplicated().sum())
print("Test duplicates:", test_df["combined_text"].duplicated().sum())

Train duplicates: 0
Test duplicates: 0


In [3]:
# Remove duplicate emails from train and test
train_df = train_df.drop_duplicates(subset="combined_text").reset_index(drop=True)
test_df = test_df.drop_duplicates(subset="combined_text").reset_index(drop=True)

print("Train shape after removing duplicates:", train_df.shape)
print("Test shape after removing duplicates:", test_df.shape)

print("Train duplicates:", train_df["combined_text"].duplicated().sum())
print("Test duplicates:", test_df["combined_text"].duplicated().sum())

Train shape after removing duplicates: (27246, 16)
Test shape after removing duplicates: (6812, 16)
Train duplicates: 0
Test duplicates: 0


In [4]:
# check overlap

# Check duplicates within each dataset
print("Train duplicates:", train_df["combined_text"].duplicated().sum())
print("Test duplicates :", test_df["combined_text"].duplicated().sum())

# Check overlap between train and test
train_texts = set(train_df["combined_text"])
test_texts = set(test_df["combined_text"])

overlap = train_texts.intersection(test_texts)

print("Train-Test duplicated texts:", len(overlap))

#Delete overlapp
df = pd.concat([train_df, test_df], ignore_index=True)

# Remove exact duplicate emails
df = df.drop_duplicates(subset="combined_text")

# Create a fresh clean split
from sklearn.model_selection import train_test_split

train_clean, test_clean = train_test_split(
    df,
    test_size=0.20,
    stratify=df["label"],
    random_state=42
)

Train duplicates: 0
Test duplicates : 0
Train-Test duplicated texts: 0


In [5]:
# check class balance
print("Train class distribution:")
print(train_df["label"].value_counts())

print("\nTest class distribution:")
print(test_df["label"].value_counts())

Train class distribution:
label
0    13774
1    13472
Name: count, dtype: int64

Test class distribution:
label
0    3444
1    3368
Name: count, dtype: int64


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack, csr_matrix


# X and y


X = train_df.drop(columns=["label"])
y = train_df["label"]

X_test_raw = test_df.drop(columns=["label"])
y_test = test_df["label"]



# Train / Validation split


X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)



# Text + numerical columns


TEXT_COL = "combined_text"

NUMERIC_COLS = [
    col for col in X.columns
    if col != TEXT_COL
]



# TF-IDF


tfidf = TfidfVectorizer(
    max_features=300,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

X_text_train = tfidf.fit_transform(X_train_raw[TEXT_COL])
X_text_val = tfidf.transform(X_val_raw[TEXT_COL])
X_text_test = tfidf.transform(X_test_raw[TEXT_COL])


# scale
scaler = StandardScaler()

X_num_train = scaler.fit_transform(X_train_raw[NUMERIC_COLS])
X_num_val = scaler.transform(X_val_raw[NUMERIC_COLS])
X_num_test = scaler.transform(X_test_raw[NUMERIC_COLS])


# Combine ALL features


X_train = hstack([
    X_text_train,
    csr_matrix(X_num_train)
])

X_val = hstack([
    X_text_val,
    csr_matrix(X_num_val)
])

X_test = hstack([
    X_text_test,
    csr_matrix(X_num_test)
])


print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Training: (21796, 314)
Validation: (5450, 314)
Test: (6812, 314)


In [9]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

print("Logistic Regression trained!")


Logistic Regression trained!


In [10]:
from sklearn.metrics import accuracy_score, recall_score, f1_score

train_pred = lr.predict(X_train)
test_pred = lr.predict(X_test)

print("Train Accuracy:", accuracy_score(y_train, train_pred))
print("Test Accuracy :", accuracy_score(y_test, test_pred))

print("Train Recall:", recall_score(y_train, train_pred))
print("Test Recall :", recall_score(y_test, test_pred))

print("Train F1:", f1_score(y_train, train_pred))
print("Test F1 :", f1_score(y_test, test_pred))


Train Accuracy: 0.9820150486327767
Test Accuracy : 0.9815032295948326
Train Recall: 0.9861742599981442
Test Recall : 0.9869358669833729
Train F1: 0.9818920916481892
Test F1 : 0.9813994685562445


In [76]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, f1_score

lr_c01 = LogisticRegression(C=0.1, max_iter=1000)
lr_c01.fit(X_train, y_train)

train_pred = lr_c01.predict(X_train)
test_pred = lr_c01.predict(X_test)

print("Train Accuracy:", accuracy_score(y_train, train_pred))
print("Test Accuracy :", accuracy_score(y_test, test_pred))
print("Train Recall:", recall_score(y_train, train_pred))
print("Test Recall :", recall_score(y_test, test_pred))
print("Train F1:", f1_score(y_train, train_pred))
print("Test F1:", f1_score(y_test, test_pred))

Train Accuracy: 0.9664003502243624
Test Accuracy : 0.9667519181585678
Train Recall: 0.9668159459657881
Test Recall : 0.9655172413793104
Train F1: 0.9662129938735831
Test F1: 0.9656672362901974


## Cross-validated Logistic Regression

Instead of hand-tuning `C`, search the hyperparameters with stratified k-fold CV.
`StandardScaler(with_mean=False)` sits inside the pipeline so it is re-fit on each
training fold only (no leakage), and it puts the raw count features
(`capital_letter_count`, `special_char_count`, ...) on the same scale as TF-IDF,
which is what was causing the `ConvergenceWarning`.

In [12]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold

pipe = Pipeline([
    # with_mean=False keeps the matrix sparse
    ("scale", StandardScaler(with_mean=False)),
    ("clf", LogisticRegression(max_iter=2000, solver="liblinear")),
])

param_grid = [
    {
        "clf__C": [0.01, 0.1, 0.5, 1, 5, 10, 50],
        # l1_ratio replaces the deprecated `penalty` arg: 0.0 = L2, 1.0 = L1
        "clf__l1_ratio": [0.0, 1.0],
        "clf__class_weight": [None, "balanced"],
    },
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    pipe,
    param_grid,
    scoring={"f1": "f1", "recall": "recall", "accuracy": "accuracy"},
    refit="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=True,
    verbose=1,
)

grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV F1 : %.4f" % grid.best_score_)

Fitting 5 folds for each of 28 candidates, totalling 140 fits
Best params: {'clf__C': 0.5, 'clf__class_weight': None, 'clf__l1_ratio': 1.0}
Best CV F1 : 0.9820


In [13]:
# Full CV results, best first
cv_results = pd.DataFrame(grid.cv_results_)

cols = [
    "param_clf__C", "param_clf__l1_ratio", "param_clf__class_weight",
    "mean_train_f1", "mean_test_f1", "std_test_f1",
    "mean_test_recall", "mean_test_accuracy",
]

print(
    cv_results[cols]
    .sort_values("mean_test_f1", ascending=False)
    .head(10)
    .to_string(index=False)
)

 param_clf__C  param_clf__l1_ratio param_clf__class_weight  mean_train_f1  mean_test_f1  std_test_f1  mean_test_recall  mean_test_accuracy
          0.5                  1.0                    None       0.987679      0.981979     0.001488          0.986081            0.982107
          0.5                  1.0                balanced       0.987600      0.981888     0.001468          0.986081            0.982015
          0.1                  1.0                    None       0.986106      0.981662     0.001915          0.986081            0.981786
          0.1                  1.0                balanced       0.986120      0.981618     0.001867          0.986174            0.981740
          1.0                  1.0                balanced       0.988029      0.981606     0.001370          0.985524            0.981740
          5.0                  1.0                    None       0.988523      0.981606     0.001803          0.985524            0.981740
         10.0              

In [14]:
from sklearn.metrics import accuracy_score, recall_score, f1_score, classification_report

best_lr = grid.best_estimator_

for name, X_, y_ in [
    ("Train", X_train, y_train),
    ("Val  ", X_val, y_val),
    ("Test ", X_test, y_test),
]:
    pred = best_lr.predict(X_)
    print("%s  acc=%.4f  recall=%.4f  f1=%.4f" % (
        name,
        accuracy_score(y_, pred),
        recall_score(y_, pred),
        f1_score(y_, pred),
    ))

print()
print(classification_report(y_test, best_lr.predict(X_test), target_names=["ham", "spam"]))

Train  acc=0.9877  recall=0.9904  f1=0.9876
Val    acc=0.9840  recall=0.9889  f1=0.9839
Test   acc=0.9828  recall=0.9878  f1=0.9827

              precision    recall  f1-score   support

         ham       0.99      0.98      0.98      3444
        spam       0.98      0.99      0.98      3368

    accuracy                           0.98      6812
   macro avg       0.98      0.98      0.98      6812
weighted avg       0.98      0.98      0.98      6812

